# **Instalações e Importações**

In [ ]:
%%capture

# Bibliotecas
!pip install shap scikit-survival lifelines

In [ ]:
# Importações
seed = 1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

import scipy.stats as stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.metrics import integrated_brier_score, cumulative_dynamic_auc
from sksurv.util import Surv

# **Funções**

In [ ]:
def train_preprocessing(df, ohe_encoder=None):
    """
    Preprocesses the dataset for machine learning model training.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the data to be preprocessed.
    ohe_encoder : list, optional
        List of categorical columns to be transformed with One-Hot Encoding.
        If `None`, this step will be skipped. Default is `None`.

    Returns:
    --------
    df_aux : numpy.ndarray
        Numpy matrix containing the preprocessed and normalized data.
    enc : dict
        Dictionary containing the encoders used for categorical columns.
        May include `OneHotEncoder` and `OrdinalEncoder`.
    feat_cols : pandas.Index
        List of columns/features present after preprocessing.

    Notes:
    ------
    - Columns specified in `ohe_encoder` are transformed using `OneHotEncoder`.
    - Remaining categorical columns are encoded with `OrdinalEncoder`.
    - Normalization is applied to all columns after categorical encoding.
    """
    # Data Transformation: One-Hot Encoding, Ordinal Encoding, and Normalization
    df_aux = df.copy()  # Create a copy of the input DataFrame

    enc = dict()  # Initialize a dictionary to store encoders

    # Apply One-Hot Encoding (if specified)
    if ohe_encoder is not None:
        for col in ohe_encoder:  # Iterate through columns to be one-hot encoded
            enc[col] = OneHotEncoder(handle_unknown='ignore', drop='first')  # Initialize and fit OneHotEncoder
            ohe_results = enc[col].fit_transform(df_aux[[col]])  # Apply one-hot encoding and transform
            df1 = pd.DataFrame(ohe_results.toarray(),  # Create DataFrame from encoded results
                            columns=enc[col].get_feature_names_out(),
                            index=df_aux[col].index)
            df_aux = df_aux.merge(df1, how='left', left_index=True, right_index=True)  # Merge encoded columns into main DataFrame

        df_aux.drop(columns=ohe_encoder, inplace=True)  # Drop original one-hot encoded columns

    # Apply Ordinal Encoding to remaining categorical columns
    list_categorical = df_aux.select_dtypes(include='object').columns  # Identify categorical columns
    for col in list_categorical:  # Iterate through categorical columns
        enc[col] = OrdinalEncoder(handle_unknown='use_encoded_value',  # Initialize and fit OrdinalEncoder
                                  unknown_value=-1)  # Handle unknown values by assigning -1
        df_aux[col] = enc[col].fit_transform(df_aux[[col]])  # Apply ordinal encoding and transform

    feat_cols = df_aux.columns  # Store the feature names after encoding

    return df_aux, enc, feat_cols  # Return the transformed DataFrame, encoders, normalizer, and feature names

################################################################################

def test_preprocessing(df, enc, ohe_encoder=None):
    """
    Preprocesses the test dataset using the transformers fitted on the training set.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing the test data to be preprocessed.
    enc : dict
        Dictionary containing the encoders fitted on the training set.
        May include `OneHotEncoder` and `OrdinalEncoder` for categorical columns.
    ohe_encoder : list, optional
        List of categorical columns that were transformed with One-Hot Encoding during training.
        If `None`, this step will be skipped. Default is `None`.

    Returns:
    --------
    df_aux : numpy.ndarray
        Numpy matrix containing the preprocessed and normalized test data.

    Notes:
    ------
    - Columns specified in `ohe_encoder` are transformed using the `OneHotEncoder` fitted on the training set.
    - Remaining categorical columns are transformed using the `OrdinalEncoder` fitted on training.
    - It's important to ensure that `enc` and `norm` are the same ones used in preprocessing the training set.
    """
    # Data Transformation: One-Hot Encoding and Normalization
    df_aux = df.copy()  # Create a copy of the input DataFrame

    # Apply One-Hot Encoding (if encoder is provided)
    if ohe_encoder is not None:
        for col in ohe_encoder: # Iterate through categorical columns to be encoded
            ohe_results = enc[col].transform(df_aux[[col]])  # Apply one-hot encoding
            df1 = pd.DataFrame(ohe_results.toarray(),  # Create a new DataFrame with the encoded features
                            columns=enc[col].get_feature_names_out(),
                            index=df_aux[col].index)
            df_aux = df_aux.merge(df1, how='left', left_index=True, right_index=True)  # Merge encoded features into the main DataFrame

        df_aux.drop(columns=ohe_encoder, inplace=True)  # Remove original categorical columns

    # Apply Encoding to other categorical features
    list_categorical = df_aux.select_dtypes(include='object').columns #Get the list of categorical columns
    for col in list_categorical: #Iterate through the categorical columns
        df_aux[col] = enc[col].transform(df_aux[[col]]) #Apply the transformation

    return df_aux  # Return the transformed DataFrame

################################################################################

def preprocessing(x_train, x_test, ohe_encoder=None, return_enc_norm=False,
                  random_state=0):
    """
    Preprocesses both training and test datasets, including category encoding and normalization.

    Parameters:
    -----------
    x_train : pandas.DataFrame
        Training dataset.
    x_test : pandas.DataFrame
        Test dataset.
    ohe_encoder : list, optional
        List of categorical columns to be transformed with One-Hot Encoding.
        If `None`, this step will be skipped. Default is `None`.
    return_enc_norm : bool, optional
        If `True`, also returns the fitted `enc` (encoders) and `norm` (normalizer) objects.
        Default is `False`.
    random_state : int, optional
        Seed to ensure reproducibility in preprocessing.
        Default is `0`.

    Returns:
    --------
    X_train_ : numpy.ndarray
        Numpy matrix containing the preprocessed and normalized training data.
    X_test_ : numpy.ndarray
        Numpy matrix containing the preprocessed and normalized test data.
    feat_cols : pandas.Index
        List of columns/features present after preprocessing.
    enc : dict, optional
        Dictionary containing the encoders fitted on the training set (returned if `return_enc_norm=True`).

    Notes:
    ------
    - The function internally uses `train_preprocessing` to fit the encoders and normalizers on the training set.
    - The test set is transformed using the same encoders and normalizers fitted on training.
    - If `return_enc_norm=False`, only the preprocessed data and feature columns are returned.
    """
    # Data Preprocessing
    # Apply preprocessing steps to the training and testing data
    X_train_, enc, feat_cols = train_preprocessing(x_train, ohe_encoder=ohe_encoder)
    X_test_ = test_preprocessing(x_test, enc, ohe_encoder=ohe_encoder)

    # Return preprocessed data and preprocessing objects (encoder and normalizer) if requested
    if return_enc_norm:
        return X_train_, X_test_, feat_cols, enc
    else:
        return X_train_, X_test_, feat_cols

# **Treino e Teste**



In [ ]:
# Dataset de câncer colorretal
!gdown 1j7T40CUsWwMJvFJqdGoHI5PCHFwk-NO_ --quiet

In [ ]:
# Leitura dos dados
df_colo = pd.read_csv('colorretal.csv')
print(df_colo.shape)  # Dimensões do conjunto de dados
df_colo.head(3)  # 3 primeiras linhas do conjunto de dados

(44857, 49)


,INSTITU,ESCOLARI,IDADE,SEXO,IBGE,CATEATEND,DIAGPREV,TOPO,EC,ECGRUP,...,TRATCONS_CAT,DIAGTRAT_CAT,PRESENCA_META,PRESENCA_REC,obito_geral,obito_cancer,sobrevida_ano1,sobrevida_ano3,sobrevida_ano5,meses_diag
0,612374,9,91,2,3550308,2,2,C209,IIA,II,...,0,1,0,0,0,0,1,0,0,23
1,8,9,79,2,3550308,9,1,C180,II,II,...,0,0,0,0,0,0,1,1,1,61
2,8,9,59,2,3534401,9,2,C209,II,II,...,0,0,0,1,0,0,1,1,1,61


In [ ]:
# Colunas
df_colo.columns

Index(['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'IBGE', 'CATEATEND', 'DIAGPREV',
       'TOPO', 'EC', 'ECGRUP', 'NENHUM', 'CIRURGIA', 'RADIO', 'QUIMIO',
       'HORMONIO', 'IMUNO', 'OUTROS', 'NENHUMAPOS', 'CIRURAPOS', 'RADIOAPOS',
       'QUIMIOAPOS', 'HORMOAPOS', 'IMUNOAPOS', 'OUTROAPOS', 'ULTINFO',
       'CONSDIAG', 'TRATCONS', 'DIAGTRAT', 'ANODIAG', 'DRS', 'RRAS', 'DSCINST',
       'IBGEATEN', 'HABILIT2', 'DRS_INST', 'RRAS_INST', 'ECGRUP_CAT',
       'ULTIDIAG', 'CONSDIAG_CAT', 'TRATCONS_CAT', 'DIAGTRAT_CAT',
       'PRESENCA_META', 'PRESENCA_REC', 'obito_geral', 'obito_cancer',
       'sobrevida_ano1', 'sobrevida_ano3', 'sobrevida_ano5', 'meses_diag'],
      dtype='object')

In [ ]:
# Distribuição dos dados de óbito por qualquer razão
df_colo['obito_geral'].value_counts()

,count
obito_geral,
0,22980
1,21877


In [ ]:
# Divisão dos dados em treino e teste
# Separação das features (X) e label (y)
X = df_colo.drop(['obito_geral'], axis=1)  # Features (todas as colunas, exceto 'obito_geral')
y = df_colo['obito_geral']  # Label ('obito_geral')

# Divisão do conjunto de dados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,  # 80% treino, 20% teste
                                                    random_state=seed,  # Reprodutibilidade
                                                    stratify=y)  # Estratificação para manter a proporção das classes

# Tamanhos dos conjuntos após a divisão dos dados
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (35885, 48)
y_train: (35885,)
X_test: (8972, 48)
y_test: (8972,)


In [ ]:
# Salvamento dos datasets
# Cancatenação das features e label
df_treino = pd.concat([X_train, y_train], axis=1)  # Combinação do X_train e y_train em um conjunto de dados
df_teste = pd.concat([X_test, y_test], axis=1)  # Combinação do X_test e y_test em um conjunto de dados

df_treino.to_csv('colorretal_treino.csv', index=False)  # Salva o dataset de treino como 'colorretal_treino.csv'
df_teste.to_csv('colorretal_teste.csv', index=False)  # Salva o dataset de teste como 'colorretal_teste.csv'

# **Modelo**

In [ ]:
df_treino = pd.read_csv('colorretal_treino.csv')
df_teste = pd.read_csv('colorretal_teste.csv')

df_treino.shape, df_teste.shape

((35885, 49), (8972, 49))

In [ ]:
# Exclusão das linhas com ULTIDIAG negativo
df_treino = df_treino[df_treino.ULTIDIAG >= 0]

df_treino.shape

(35884, 49)

In [ ]:
# Retirada das colunas que não serão utilizadas como features nos modelos de sobrevida
list_drop = ['obito_cancer', 'sobrevida_ano1', 'sobrevida_ano3', 'sobrevida_ano5',
             'ECGRUP', 'NENHUM', 'CIRURGIA', 'RADIO', 'QUIMIO', 'HORMONIO',
             'IMUNO', 'OUTROS', 'NENHUMAPOS', 'CIRURAPOS', 'RADIOAPOS',
             'QUIMIOAPOS', 'HORMOAPOS', 'IMUNOAPOS', 'OUTROAPOS', 'ULTINFO',
             'CONSDIAG', 'TRATCONS', 'DIAGTRAT', 'RRAS', 'DSCINST', 'RRAS_INST',
             'ECGRUP_CAT', 'ULTIDIAG', 'CONSDIAG_CAT', 'PRESENCA_META',
             'PRESENCA_REC']

X_train = df_treino.drop(list_drop, axis=1)  # Conjunto de treino

X_test = df_teste.drop(list_drop, axis=1)  # Conjunto de validação

X_train.shape, X_test.shape

((35884, 18), (8972, 18))

## **Testes Estatísticos**

Você pode realizar esses testes estatísticos em Python utilizando as bibliotecas SciPy, statsmodels e scikit-posthocs. Vou detalhar as bibliotecas para cada um dos testes:

1. Shapiro-Wilk (testa a normalidade dos dados):
    * Função: `scipy.stats.shapiro`

2. T-Student (compara as médias de dois grupos para verificar se são estatisticamente diferentes):
    * Função: `scipy.stats.ttest_ind` (para amostras independentes) ou `scipy.stats.ttest_rel` (para amostras pareadas)

3. Qui-Quadrado (testa a independência entre variáveis categóricas):
    * Função: `scipy.stats.chi2_contingency`

4. Teste Exato de Fisher (testa a associação entre variáveis categóricas em tabelas pequenas, geralmente 2x2):
    * Função: `scipy.stats.fisher_exact`

5. Mann-Whitney U (teste não paramétrico para comparar duas amostras independentes):
    * Função: `scipy.stats.mannwhitneyu`

In [ ]:
data = X_train.copy()
data.shape

(35884, 18)

In [ ]:
data.columns

Index(['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'IBGE', 'CATEATEND', 'DIAGPREV',
       'TOPO', 'EC', 'ANODIAG', 'DRS', 'IBGEATEN', 'HABILIT2', 'DRS_INST',
       'TRATCONS_CAT', 'DIAGTRAT_CAT', 'meses_diag', 'obito_geral'],
      dtype='object')

In [ ]:
# Se o valor for menor que o nível de significância escolhido (por exemplo, 0,05),
# indica que há uma associação significativa entre as colunas escolhidas.
cat = ['INSTITU', 'ESCOLARI', 'SEXO', 'IBGE', 'CATEATEND', 'DIAGPREV',
       'TOPO', 'EC', 'DRS', 'IBGEATEN', 'HABILIT2', 'DRS_INST',
       'TRATCONS_CAT', 'DIAGTRAT_CAT']

for col in cat:

    # Criar a tabela de contingência
    contingency_table = pd.crosstab(data[col], data.obito_geral)

    # Categóricas
    chi2_stat, chi2_p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f'{col} -> Qui-Quadrado Teste: {chi2_stat:.6f}, p-value: {chi2_p:.6f}')

INSTITU -> Qui-Quadrado Teste: 2011.308531, p-value: 0.000000
ESCOLARI -> Qui-Quadrado Teste: 426.543630, p-value: 0.000000
SEXO -> Qui-Quadrado Teste: 72.299350, p-value: 0.000000
IBGE -> Qui-Quadrado Teste: 1127.426679, p-value: 0.000000
CATEATEND -> Qui-Quadrado Teste: 1160.840680, p-value: 0.000000
DIAGPREV -> Qui-Quadrado Teste: 275.125321, p-value: 0.000000
TOPO -> Qui-Quadrado Teste: 144.935751, p-value: 0.000000
EC -> Qui-Quadrado Teste: 7601.616852, p-value: 0.000000
DRS -> Qui-Quadrado Teste: 240.761604, p-value: 0.000000
IBGEATEN -> Qui-Quadrado Teste: 818.260455, p-value: 0.000000
HABILIT2 -> Qui-Quadrado Teste: 58.625486, p-value: 0.000000
DRS_INST -> Qui-Quadrado Teste: 290.194543, p-value: 0.000000
TRATCONS_CAT -> Qui-Quadrado Teste: 1017.932808, p-value: 0.000000
DIAGTRAT_CAT -> Qui-Quadrado Teste: 1131.691043, p-value: 0.000000


In [ ]:
# Se p-value > 0.05, não rejeitamos a hipótese nula (não temos evidências suficientes para afirmar que os dados não são normais).
# Se p-value ≤ 0.05, rejeitamos a hipótese nula (indicando que os dados provavelmente não seguem uma distribuição normal).
cont = ['IDADE', 'ANODIAG', 'meses_diag']

for col in cont:
    # Teste Shapiro-Wilk (normalidade)
    shapiro_stat, shapiro_p = stats.shapiro(data[col])
    print(f'{col} -> Shapiro-Wilk Teste: {shapiro_stat:.6f}, p-value: {shapiro_p:.6f}')

IDADE -> Shapiro-Wilk Teste: 0.992343, p-value: 0.000000
ANODIAG -> Shapiro-Wilk Teste: 0.959595, p-value: 0.000000
meses_diag -> Shapiro-Wilk Teste: 0.858147, p-value: 0.000000


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 35884.
  res = hypotest_fun_out(*samples, **kwds)


In [ ]:
# # Se normal:
# t_stat, t_p = stats.ttest_ind(data1, data2)
# print(f"T-Student Teste: {t_stat}, p-value: {t_p}")

In [ ]:
# Se p-value > 0.05, não rejeitamos a hipótese nula (não há diferença significativa entre os grupos).
# Se p-value ≤ 0.05, rejeitamos a hipótese nula (indicando que há uma diferença significativa entre os grupos).
# Se Shapiro não indica uma distribuidção normal:
for c in cont:

    data1 = data[data.obito_geral == 1][c]
    data2 = data[data.obito_geral == 0][c]
    mannwhitney_stat, mannwhitney_p = stats.mannwhitneyu(data1, data2)
    print(f"{c} -> Mann-Whitney Teste: {mannwhitney_stat}, p-value: {mannwhitney_p:.6f}")

IDADE -> Mann-Whitney Teste: 183581265.0, p-value: 0.000000
ANODIAG -> Mann-Whitney Teste: 130672434.0, p-value: 0.000000
meses_diag -> Mann-Whitney Teste: 58914466.5, p-value: 0.000000


## **Cox**

In [ ]:
X_train.isna().sum().sort_values(ascending=False).head(5)

,0
INSTITU,0
ESCOLARI,0
IDADE,0
SEXO,0
IBGE,0


In [ ]:
# Pré-processamento dos conjuntos de treino e teste
X_train, X_test, feat_cols, enc = preprocessing(X_train, X_test,
                                                return_enc_norm=True,  # Retornar os modelos de codificação e normalização
                                                random_state=seed)  # Reprodutibilidade

# Tamanho final dos dados de treinamento e teste
X_train.shape, X_test.shape

((35884, 18), (8972, 18))

In [ ]:
# Features selecionadas
feat_cols

Index(['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'IBGE', 'CATEATEND', 'DIAGPREV',
       'TOPO', 'EC', 'ANODIAG', 'DRS', 'IBGEATEN', 'HABILIT2', 'DRS_INST',
       'TRATCONS_CAT', 'DIAGTRAT_CAT', 'meses_diag', 'obito_geral'],
      dtype='object')

**Cox Uni**

In [ ]:
X_test.columns

Index(['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'IBGE', 'CATEATEND', 'DIAGPREV',
       'TOPO', 'EC', 'ANODIAG', 'DRS', 'IBGEATEN', 'HABILIT2', 'DRS_INST',
       'TRATCONS_CAT', 'DIAGTRAT_CAT', 'meses_diag', 'obito_geral'],
      dtype='object')

In [ ]:
# Define as colunas que serão removidas dos conjuntos de atributos
drop_cols = ['IBGE', 'IBGEATEN']

# Remove as variáveis de identificação geográfica dos conjuntos de treinamento e teste
X_train_feats = X_train.drop(columns=drop_cols)
X_test_feats = X_test.drop(columns=drop_cols)

# Exibe as colunas de atributos que permanecem no conjunto de treinamento
X_train_feats.columns

Index(['INSTITU', 'ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO',
       'EC', 'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'TRATCONS_CAT',
       'DIAGTRAT_CAT', 'meses_diag', 'obito_geral'],
      dtype='object')

In [ ]:
cat = {'ESCOLARI': 5, 'SEXO': 2, 'CATEATEND': 1, 'DIAGPREV': 1, 'TOPO': 11,
       'EC': 0, 'HABILIT2': 2, 'TRATCONS_CAT': 0, 'DIAGTRAT_CAT': 0, 'DRS': 1,
       'DRS_INST': 1}

for col, ref in cat.items():
    print(f'{col}')

    # Fórmula com especificação de categoria de referência
    formula = f"C({col}, Treatment(reference={ref}))"

    # Modelo de Cox (via statsmodels)
    cox = CoxPHFitter()
    cox.fit(X_train_feats, duration_col="meses_diag", event_col="obito_geral",
            formula=formula)

    # Sumário
    print(cox.summary)
    print('\n\n')

ESCOLARI
                                              coef  exp(coef)  se(coef)  \
covariate                                                                 
C(ESCOLARI, Treatment(reference=5))[T.1]  0.670911   1.956019  0.047673   
C(ESCOLARI, Treatment(reference=5))[T.2]  0.460373   1.584665  0.035297   
C(ESCOLARI, Treatment(reference=5))[T.3]  0.406581   1.501675  0.036662   
C(ESCOLARI, Treatment(reference=5))[T.4]  0.240661   1.272089  0.038801   
C(ESCOLARI, Treatment(reference=5))[T.9]  0.441430   1.554929  0.034972   

                                          coef lower 95%  coef upper 95%  \
covariate                                                                  
C(ESCOLARI, Treatment(reference=5))[T.1]        0.577474        0.764349   
C(ESCOLARI, Treatment(reference=5))[T.2]        0.391193        0.529553   
C(ESCOLARI, Treatment(reference=5))[T.3]        0.334724        0.478438   
C(ESCOLARI, Treatment(reference=5))[T.4]        0.164612        0.316709   
C(ESCOLAR

In [ ]:
cont = ['IDADE', 'ANODIAG']

for col in cont:
    print(f'{col}')

    # Fórmula com especificação de categoria de referência
    formula = f"{col}"

    # Modelo de Cox (via statsmodels)
    cox = CoxPHFitter()
    cox.fit(X_train_feats, duration_col="meses_diag", event_col="obito_geral",
            formula=formula)

    # Sumário
    print(cox.summary)
    print('\n\n')

IDADE
               coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  \
covariate                                                                  
IDADE      0.015935   1.016063  0.000621        0.014718        0.017152   

           exp(coef) lower 95%  exp(coef) upper 95%  cmp to         z  \
covariate                                                               
IDADE                 1.014827               1.0173     0.0  25.66546   

                       p    -log2(p)  
covariate                             
IDADE      2.841888e-145  480.172724  



ANODIAG
               coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  \
covariate                                                                  
ANODIAG   -0.005678   0.994338  0.001267       -0.008162       -0.003195   

           exp(coef) lower 95%  exp(coef) upper 95%  cmp to         z  \
covariate                                                               
ANODIAG               0.991871            

**Cox Multi**

O Backward Elimination baseado no teste de razão de verossimilhança (LR Test) pode ser implementado para modelos de Cox. O teste compara a qualidade de ajuste entre dois modelos aninhados (um modelo completo e outro com uma variável removida) usando a estatística de razão de verossimilhança (−2Δlog-likelihood).

In [ ]:
# Define as categorias de referência para as variáveis categóricas
# que serão utilizadas posteriormente na modelagem estatística
ref_dict = {
    'ESCOLARI': 5,
    'SEXO': 2,
    'CATEATEND': 1,
    'DIAGPREV': 1,
    'TOPO': 11,
    'EC': 0,
    'DRS': 1,
    'HABILIT2': 2,
    'DRS_INST': 1,
    #'TRATCONS_CAT': 0,
    'DIAGTRAT_CAT': 0
}

# Define a lista de variáveis selecionadas para análise
# As variáveis categóricas possuem categorias de referência
# definidas no dicionário acima
cols_sig = [
    'ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC',
    'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', #'TRATCONS_CAT',
    'DIAGTRAT_CAT'
]

In [ ]:
# Cria uma lista vazia para armazenar as variáveis que serão testadas no modelo
vars_teste = []

# Percorre todas as variáveis selecionadas para a análise
for var in cols_sig:
    # Adiciona a variável à lista de variáveis a serem testadas
    vars_teste.append(var)


try:
    # Cria uma lista vazia para armazenar os termos da fórmula do modelo
    terms = []

    # Percorre as variáveis selecionadas para o modelo
    for v in vars_teste:

        # Verifica se a variável possui uma categoria de referência
        # definida no dicionário ref_dict
        if v in ref_dict:
            # Para variáveis categóricas, utiliza a codificação de tratamento
            # com a categoria especificada como referência
            terms.append(
                f"C({v}, Treatment(reference={ref_dict[v]}))"
            )

        else:
            # Para variáveis que não possuem categoria de referência definida,
            # adiciona a variável diretamente à fórmula
            terms.append(v)

    # Constrói a fórmula do modelo de riscos proporcionais de Cox,
    # utilizando 'meses_diag' como variável de tempo até o evento
    formula = f"meses_diag ~ {' + '.join(terms)}"

    # Ajusta o modelo de riscos proporcionais de Cox
    # utilizando a fórmula construída e os dados de treinamento
    smf.phreg(
        formula=formula,
        data=X_train_feats,
        status=X_train_feats["obito_geral"]
    ).fit()

    # Informa que o modelo foi ajustado com sucesso
    # para o conjunto atual de variáveis
    print(f"OK: {vars_teste}")

except Exception as e:
    # Caso ocorra algum erro durante o ajuste do modelo,
    # exibe a variável que estava sendo adicionada
    print(f"\nERRO ao adicionar {var}")

    # Exibe todas as variáveis incluídas no modelo no momento do erro
    print(f"Variáveis no modelo: {vars_teste}")

    # Exibe a mensagem de erro retornada pelo modelo
    print(f"Motivo: {e}")

    # Interrompe o processo caso seja identificado um erro
    break

OK: ['ESCOLARI']
OK: ['ESCOLARI', 'IDADE']
OK: ['ESCOLARI', 'IDADE', 'SEXO']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS', 'HABILIT2']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST']
OK: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'DIAGTRAT_CAT']


In [ ]:
# Importa a biblioteca Patsy, utilizada para construir matrizes de desenho
# (design matrices) a partir de fórmulas estatísticas
import patsy


# Define a fórmula do modelo, especificando a variável de tempo e as
# variáveis explicativas. As variáveis categóricas são tratadas como
# fatores e possuem categorias de referência previamente definidas.
formula = """
meses_diag ~
C(ESCOLARI, Treatment(reference=5)) +
C(SEXO, Treatment(reference=2)) +
C(CATEATEND, Treatment(reference=1)) +
C(DIAGPREV, Treatment(reference=1)) +
C(TOPO, Treatment(reference=11)) +
C(EC, Treatment(reference=0)) +
C(DRS, Treatment(reference=1)) +
C(HABILIT2, Treatment(reference=2)) +
C(DRS_INST, Treatment(reference=1)) +
C(TRATCONS_CAT, Treatment(reference=0)) +
C(DIAGTRAT_CAT, Treatment(reference=0))
"""

# Constrói as matrizes de resposta (y) e de atributos (X) a partir da fórmula.
# O argumento return_type='dataframe' faz com que as matrizes sejam retornadas
# como DataFrames do pandas, facilitando sua inspeção e manipulação.
y, X = patsy.dmatrices(
    formula,
    data=X_train_feats,
    return_type='dataframe'
)

# Exibe as dimensões da matriz de atributos, indicando o número de observações
# e de colunas geradas após a codificação das variáveis categóricas
print(X.shape)

# Calcula e exibe o posto (rank) da matriz de atributos.
# O posto indica o número de colunas linearmente independentes e permite
# verificar a existência de dependência linear entre as variáveis do modelo.
print(np.linalg.matrix_rank(X))

(35884, 76)
75


In [ ]:
# Importa a função utilizada para calcular o Fator de Inflação da Variância (VIF),
# uma medida utilizada para avaliar a multicolinearidade entre as variáveis
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Cria um DataFrame para armazenar o nome de cada variável e seu respectivo VIF
vif = pd.DataFrame({
    "variavel": X.columns,

    # Calcula o VIF para cada coluna da matriz de atributos X.
    # O VIF quantifica o grau de multicolinearidade de cada variável
    # em relação às demais variáveis do modelo.
    "VIF": [variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])]
})

# Ordena as variáveis pelo VIF em ordem decrescente e exibe as 20 variáveis
# com os maiores valores de multicolinearidade
vif.sort_values("VIF", ascending=False).head(20)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variavel,VIF
75,"C(DIAGTRAT_CAT, Treatment(reference=0))[T.3]",inf
72,"C(TRATCONS_CAT, Treatment(reference=0))[T.3]",inf
0,Intercept,35.755150
69,"C(DRS_INST, Treatment(reference=1))[T.17]",11.818548
49,"C(DRS, Treatment(reference=1))[T.17]",11.583400
59,"C(DRS_INST, Treatment(reference=1))[T.7]",9.125829
39,"C(DRS, Treatment(reference=1))[T.7]",8.920771
62,"C(DRS_INST, Treatment(reference=1))[T.10]",8.057421
42,"C(DRS, Treatment(reference=1))[T.10]",7.767344
63,"C(DRS_INST, Treatment(reference=1))[T.11]",7.683490


In [ ]:
# Importa a função utilizada para calcular o Fator de Inflação da Variância (VIF),
# uma medida utilizada para avaliar a presença de multicolinearidade entre as variáveis
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Seleciona, a partir do conjunto de treinamento, as variáveis definidas
# na lista de variáveis selecionadas para a análise
X = X_train_feats[cols_sig]

# Cria um DataFrame vazio para armazenar os resultados do cálculo do VIF
vif = pd.DataFrame()

# Armazena os nomes das variáveis selecionadas no DataFrame de resultados
vif["Variavel"] = X.columns

# Calcula o VIF para cada variável da matriz de atributos.
# O VIF avalia o grau de multicolinearidade de cada variável
# em relação às demais variáveis incluídas no conjunto de dados.
vif["VIF"] = [variance_inflation_factor(X.values, i)
              for i in range(X.shape[1])]

# Ordena os resultados pelo VIF em ordem decrescente e exibe a tabela completa,
# permitindo identificar as variáveis com maiores níveis de multicolinearidade
print(vif.sort_values("VIF", ascending=False))

        Variavel        VIF
7        ANODIAG  70.094115
1          IDADE  25.204981
4       DIAGPREV  13.852697
2           SEXO   9.849978
10      DRS_INST   9.589983
8            DRS   9.194331
5           TOPO   8.263508
9       HABILIT2   5.748477
0       ESCOLARI   3.807905
6             EC   3.126283
3      CATEATEND   2.599390
11  DIAGTRAT_CAT   1.855068


### **CoxPHFitter**

In [ ]:
def backward_elimination_lr(data, time_col, event_col, candidate_vars, ref_dict=None, alpha=0.05):
    """
    Backward elimination baseado no teste de razão de verossimilhança.
    Args:
        data (pd.DataFrame): Dados do estudo.
        time_col (str): Nome da coluna de tempo.
        event_col (str): Nome da coluna de status (evento).
        candidate_vars (list): Variáveis candidatas.
        ref_dict (dict): Dicionário com categorias de referência, ex: {'coluna': valor_referência}.
        alpha (float): Nível de significância para o teste de LR.
    Returns:
        list: Variáveis selecionadas.
    """
    selected_vars = candidate_vars[:]

    def create_formula(variables):
        """
        Cria a fórmula considerando as categorias de referência.
        """
        terms = []
        for var in variables:
            if var in ref_dict:  # Adiciona a referência para variáveis categóricas
                terms.append(f"C({var}, Treatment(reference={ref_dict[var]}))")
            else:  # Variáveis contínuas ou sem referência
                terms.append(var)
        return f"{' + '.join(terms)}"

    while len(selected_vars) > 0:
        # Modelo completo
        formula_full = create_formula(selected_vars)
        try:
            model_full = CoxPHFitter().fit(data,
                                           duration_col=time_col,
                                           event_col=event_col,
                                           formula=formula_full)
        except Exception as e:
            print("Erro no modelo completo:")
            print(selected_vars)
            raise e

        # Testar remoção de cada variável
        lr_stats = []
        for var in selected_vars:
            reduced_vars = [v for v in selected_vars if v != var]
            formula_reduced = create_formula(reduced_vars)
            try:
                model_reduced = CoxPHFitter().fit(data,
                                                  duration_col=time_col,
                                                  event_col=event_col,
                                                  formula=formula_reduced)
            except Exception as e:
                print("Erro no modelo:")
                print(reduced_vars)
                raise e

            # Estatística de razão de verossimilhança
            df_diff = len(model_full.params_) - len(model_reduced.params_)
            lr_stat = 2 * (model_full.log_likelihood_ - model_reduced.log_likelihood_)
            p_value = chi2.sf(lr_stat, df=df_diff)  # df=1 para remover uma variável
            lr_stats.append((var, lr_stat, p_value))

        # Ordenar por p-valor
        lr_stats.sort(key=lambda x: x[2], reverse=True)
        worst_var, worst_lr_stat, worst_p_value = lr_stats[0]

        # Remover variável com maior p-valor se não for significativa
        if worst_p_value > alpha:
            print(f"Removendo variável: {worst_var} (p-valor: {worst_p_value:.4f})")
            selected_vars.remove(worst_var)
        else:
            break

    return selected_vars

In [ ]:
# Define as categorias de referência para as variáveis categóricas.
# Essas categorias serão utilizadas como base de comparação na modelagem
# estatística para a interpretação dos coeficientes das demais categorias.
ref_dict = {
    'ESCOLARI': 5,
    'SEXO': 2,
    'CATEATEND': 1,
    'DIAGPREV': 1,
    'TOPO': 11,
    'EC': 0,
    'DRS': 1,
    'HABILIT2': 2,
    'DRS_INST': 1,
    #'TRATCONS_CAT': 0,
    'DIAGTRAT_CAT': 0
}

# Define a lista de variáveis selecionadas para compor o modelo.
# Inclui variáveis categóricas e numéricas que foram previamente selecionadas
# para a análise.
cols_sig = [
    'ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC',
    'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', #'TRATCONS_CAT',
    'DIAGTRAT_CAT'
]

In [ ]:
# Realiza a seleção de variáveis por meio do procedimento de eliminação
# regressiva baseado no teste de razão de verossimilhança (Likelihood Ratio),
# utilizando nível de significância de 5%.
final_vars = backward_elimination_lr(
    data=X_train_feats,
    time_col="meses_diag",
    event_col="obito_geral",
    candidate_vars=cols_sig,
    ref_dict=ref_dict,
    alpha=0.05
)

# Exibe a lista final de variáveis selecionadas pelo procedimento de
# eliminação regressiva
print("Variáveis finais selecionadas:", final_vars)

Variáveis finais selecionadas: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'DIAGTRAT_CAT']


In [ ]:
final_vars = cols_sig

In [ ]:
# Criar a fórmula final com base nas variáveis selecionadas
def create_final_formula(variables, ref_dict):
    """
    Cria a fórmula final considerando categorias de referência.
    Args:
        variables (list): Variáveis selecionadas.
        ref_dict (dict): Dicionário com categorias de referência.
    Returns:
        str: Fórmula para o modelo de Cox.
    """
    terms = []
    for var in variables:
        if var in ref_dict:  # Variáveis categóricas com referência
            terms.append(f"C({var}, Treatment(reference={ref_dict[var]}))")
        else:  # Variáveis contínuas ou sem referência
            terms.append(var)
    return f"{' + '.join(terms)}"

In [ ]:
# Gerar a fórmula final
formula_final = create_final_formula(final_vars, ref_dict)

# Ajustar o modelo final
model_final = CoxPHFitter()
model_final.fit(X_train_feats, duration_col='meses_diag',
                event_col='obito_geral', formula=formula_final)

# Exibir sumário do modelo final
model_final.print_summary()

<lifelines.CoxPHFitter: fitted with 35884 total observations, 18383 right-censored observations>
             duration col = 'meses_diag'
                event col = 'obito_geral'
      baseline estimation = breslow
   number of observations = 35884
number of events observed = 17501
   partial log-likelihood = -168553.78
         time fit was run = 2026-07-01 19:00:46 UTC

---
                                              coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                     
C(ESCOLARI, Treatment(reference=5))[T.1]      0.40      1.49      0.05            0.30            0.49                1.35                1.64
C(ESCOLARI, Treatment(reference=5))[T.2]      0.29      1.33      0.04            0.21            0.36                1.24                1.43
C(ESCOLARI, Treatment(reference=5))[T.3]      0.28      1.33      0.04            0.21            0.36                1.23                1.43
C(ESCOLARI, Treatment(reference=5))[T.4]      0.20      1.22      0.04            0.12            0.28                1.13                1.32
C(ESCOLARI, Treatment(reference=5))[T.9]      0.38      1.47      0.04            0.31            0.45                1.37                1.57
IDADE                                         0.02      1.02      0.00            0.02            0.02                1.02                1.02
C(SEXO, Treatment(reference=2))[T.1]          0.12      1.13      0.02            0.09            0.15                1.10                1.16
C(CATEATEND, Treatment(reference=1))[T.2]     0.33      1.40      0.04            0.26            0.40                1.30                1.50
C(CATEATEND, Treatment(reference=1))[T.3]    -0.13      0.88      0.14           -0.39            0.14                0.68                1.15
C(CATEATEND, Treatment(reference=1))[T.9]     0.66      1.94      0.04            0.58            0.74                1.79                2.10
C(DIAGPREV, Treatment(reference=1))[T.2]     -0.19      0.82      0.02           -0.23           -0.16                0.80                0.85
C(TOPO, Treatment(reference=11))[T.0.0]      -0.11      0.89      0.05           -0.20           -0.02                0.82                0.98
C(TOPO, Treatment(reference=11))[T.1.0]      -0.28      0.75      0.22           -0.72            0.16                0.49                1.17
C(TOPO, Treatment(reference=11))[T.2.0]      -0.11      0.90      0.03           -0.17           -0.05                0.85                0.95
C(TOPO, Treatment(reference=11))[T.3.0]       0.05      1.05      0.08           -0.11            0.20                0.90                1.23
C(TOPO, Treatment(reference=11))[T.4.0]      -0.05      0.95      0.04           -0.13            0.04                0.88                1.04
C(TOPO, Treatment(reference=11))[T.5.0]       0.05      1.06      0.08           -0.11            0.22                0.89                1.25
C(TOPO, Treatment(reference=11))[T.6.0]      -0.21      0.81      0.04           -0.28           -0.13                0.75                0.88
C(TOPO, Treatment(reference=11))[T.7.0]      -0.20      0.82      0.02           -0.25           -0.15                0.78                0.86
C(TOPO, Treatment(reference=11))[T.8.0]       0.08      1.08      0.10           -0.12            0.27                0.89                1.31
C(TOPO, Treatment(reference=11))[T.9.0]      -0.09      0.91      0.02           -0.14           -0.05                0.87                0.95
C(TOPO, Treatment(reference=11))[T.10.0]     -0.31      0.74      0.04           -0.38           -0.24                0.69                0.79
C(EC, Treatment(reference=0))[T.1.0]          0.37      1.45      0.04            0.29            0.45                1.34                1.57
C(EC, Treatment(reference=0))[T.2.0]         

In [ ]:
X_test_feats[final_vars].columns

Index(['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC',
       'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'DIAGTRAT_CAT'],
      dtype='object')

In [ ]:
# Escore de risco
risk_score = model_final.predict_partial_hazard(X_test_feats[final_vars]).values.flatten()

# C-index
cindex = concordance_index_censored(X_test_feats["obito_geral"].astype(bool),
                                    X_test_feats["meses_diag"],
                                    risk_score)[0]

# Conjunto de treino
y_train_surv = Surv.from_arrays(event=X_train_feats["obito_geral"].astype(bool),
                                time=X_train_feats["meses_diag"])

# Conjunto de teste
y_test_surv = Surv.from_arrays(event=X_test_feats["obito_geral"].astype(bool),
                               time=X_test_feats["meses_diag"])

cindex_ipcw = concordance_index_ipcw(survival_train=y_train_surv,
                                     survival_test=y_test_surv,
                                     estimate=risk_score)[0]

print(f"C-index: {cindex:.5f}")
print(f"C-index IPCW: {cindex_ipcw:.5f}")

C-index: 0.74205
C-index IPCW: 0.73441


In [ ]:
# Define o intervalo de tempo, em meses, utilizado para avaliar as previsões
# de sobrevida, considerando um período de acompanhamento de até 60 meses
times = np.arange(1, 61)

# Estima as funções de sobrevida para cada indivíduo do conjunto de teste
# utilizando o modelo final ajustado
surv_funcs = model_final.predict_survival_function(X_test_feats[final_vars])

# Transpõe as funções de sobrevida e converte os resultados para um array,
# obtendo as probabilidades de sobrevida de cada indivíduo ao longo do tempo
surv_probs = surv_funcs.T.values

# Calcula o Integrated Brier Score (IBS), que avalia a precisão das previsões
# de sobrevida ao longo de todo o período de acompanhamento.
# O argumento surv_probs[:, :-1] remove a última coluna para adequar as
# dimensões das probabilidades de sobrevida ao vetor de tempos utilizado.
ibs = integrated_brier_score(
    y_train_surv,
    y_test_surv,
    surv_probs[:, :-1],
    times
)

# Calcula a área sob a curva ROC dependente do tempo (AUC(t)),
# avaliando a capacidade discriminativa do modelo ao longo do período
# de acompanhamento.
times_auc, auc = cumulative_dynamic_auc(
    y_train_surv,
    y_test_surv,
    risk_score,
    times
)

# Exibe o Integrated Brier Score com cinco casas decimais
print(f"IBS: {ibs:.5f}")

# Exibe a AUC média dependente do tempo com cinco casas decimais
print(f"AUC média: {auc:.5f}")

IBS: 0.16127
AUC média: 0.78845


In [ ]:
# Teste de proporcionalidade calculado apenas no conjunto de treinamento
ph_test = proportional_hazard_test(
    model_final,           # modelo ajustado
    X_train_feats,         # DataSet
    time_transform='rank'  # rank dos tempos (default)
)

display(ph_test.summary)

,test_statistic,p,-log2(p)
ANODIAG,0.000309,9.859787e-01,0.020372
"C(CATEATEND, Treatment(reference=1))[T.2]",23.684921,1.134681e-06,19.749282
"C(CATEATEND, Treatment(reference=1))[T.3]",0.419829,5.170221e-01,0.951702
"C(CATEATEND, Treatment(reference=1))[T.9]",21.296735,3.934011e-06,17.955568
"C(DIAGPREV, Treatment(reference=1))[T.2]",76.722920,1.967072e-18,58.818656
...,...,...,...
"C(TOPO, Treatment(reference=11))[T.6.0]",24.165916,8.838272e-07,20.109732
"C(TOPO, Treatment(reference=11))[T.7.0]",23.139499,1.506638e-06,19.340236
"C(TOPO, Treatment(reference=11))[T.8.0]",16.799111,4.155277e-05,14.554696
"C(TOPO, Treatment(reference=11))[T.9.0]",29.685826,5.080494e-08,24.230456


In [ ]:
# Obtém o resumo dos resultados do teste de proporcionalidade dos riscos,
# realizado com base nos resíduos de Schoenfeld
summary = ph_test.summary

# Seleciona apenas as variáveis que apresentaram p-valor inferior a 0,05,
# indicando evidência estatística de violação da suposição de riscos proporcionais
violacoes = summary[summary['p'] < 0.05]

# Exibe as variáveis que apresentaram evidência de violação
# da suposição de riscos proporcionais
violacoes

,test_statistic,p,-log2(p)
"C(CATEATEND, Treatment(reference=1))[T.2]",23.684921,1.134681e-06,19.749282
"C(CATEATEND, Treatment(reference=1))[T.9]",21.296735,3.934011e-06,17.955568
"C(DIAGPREV, Treatment(reference=1))[T.2]",76.722920,1.967072e-18,58.818656
"C(DIAGTRAT_CAT, Treatment(reference=0))[T.1]",69.759653,6.698921e-17,53.728849
"C(DIAGTRAT_CAT, Treatment(reference=0))[T.2]",208.923264,2.359293e-47,154.892266
"C(DIAGTRAT_CAT, Treatment(reference=0))[T.3]",159.437320,1.501718e-36,119.002798
"C(DRS, Treatment(reference=1))[T.4]",6.177005,1.294211e-02,6.271783
"C(DRS_INST, Treatment(reference=1))[T.12]",6.066219,1.377916e-02,6.181368
"C(DRS_INST, Treatment(reference=1))[T.2]",4.724118,2.974244e-02,5.071333
"C(DRS_INST, Treatment(reference=1))[T.6]",4.532793,3.325130e-02,4.910445


In [ ]:
summary.to_csv('teste_proporcionalidade.csv')

In [ ]:
# Importa o módulo de expressões regulares, utilizado para extrair
# os nomes das variáveis a partir dos termos presentes no índice do resumo
import re

# Cria uma cópia do resumo dos resultados do teste de proporcionalidade
# para evitar alterações no DataFrame original
summary = ph_test.summary.copy()

# Define uma função para extrair o nome da variável original a partir
# dos termos gerados para as variáveis categóricas pelo Patsy
def extrair_variavel(nome):

    # Procura por padrões no formato C(variavel, ...)
    m = re.search(r'C\((.*?)\)', nome)

    # Caso o padrão seja encontrado, retorna o conteúdo entre os parênteses
    if m:
        return m.group(1)

    # Para variáveis que não possuem o padrão C(), mantém o nome original
    return nome

# Cria uma nova coluna contendo o nome da variável original,
# agrupando posteriormente diferentes coeficientes da mesma variável categórica
summary['variavel'] = summary.index.map(extrair_variavel)

# Agrupa os resultados por variável e calcula:
# - o menor p-valor entre seus coeficientes;
# - o número de coeficientes associados à variável.
# Em seguida, ordena as variáveis pelo menor p-valor.
agrupado = (
    summary
    .groupby('variavel')
    .agg(
        menor_p=('p', 'min'),
        n_coeficientes=('p', 'count')
    )
    .sort_values('menor_p')
)

# Classifica a suposição de proporcionalidade para cada variável.
# A suposição é considerada atendida quando o menor p-valor é superior a 0,05
# e violada quando pelo menos um dos coeficientes apresenta p-valor <= 0,05.
agrupado['proporcionalidade'] = (
    agrupado['menor_p'] > 0.05
).map({
    True: 'Atendida',
    False: 'Violada'
})

# Exibe o resumo agrupado dos resultados do teste de proporcionalidade
agrupado

,menor_p,n_coeficientes,proporcionalidade
variavel,,,
"DIAGTRAT_CAT, Treatment(reference=0",2.359293e-47,3,Violada
"DIAGPREV, Treatment(reference=1",1.967072e-18,1,Violada
"TOPO, Treatment(reference=11",1.404452e-15,11,Violada
"CATEATEND, Treatment(reference=1",1.134681e-06,3,Violada
IDADE,1.361742e-05,1,Violada
"SEXO, Treatment(reference=2",5.203341e-04,1,Violada
"EC, Treatment(reference=0",1.211427e-03,12,Violada
"HABILIT2, Treatment(reference=2",3.485680e-03,4,Violada
"ESCOLARI, Treatment(reference=5",9.643952e-03,5,Violada


A saída apresenta, para cada variável:

* test_statistic
* p
* -log2(p)

A hipótese nula é que a variável satisfaz a proporcionalidade dos riscos. Assim:

1. p > 0,05: não há evidências de violação;

2. p ≤ 0,05: há evidências de violação da hipótese.

In [ ]:
# Checagem das hipóteses do modelo e suas recomendações
model_final.check_assumptions(
    X_train_feats,              # DataSet
    p_value_threshold=0.05,     # Limiar para dar os alertas
    show_plots=True,            # Mostrar os gráficos
    columns=None                # Restringe a algumas colunas (lista) ou usa todas as colunas do DataSet
)

### **phreg**

In [ ]:
cat = {'ESCOLARI': 5, 'SEXO': 2, 'CATEATEND': 1, 'DIAGPREV': 1, 'TOPO': 11,
       'EC': 0, 'HABILIT2': 2, 'TRATCONS_CAT': 0, 'DIAGTRAT_CAT': 0, 'DRS': 1,
       'DRS_INST': 1}

for col, ref in cat.items():
    print(f'{col}')

    # Fórmula com especificação de categoria de referência
    formula = f"meses_diag ~ C({col}, Treatment(reference={ref}))"

    # Modelo de Cox (via statsmodels)
    model = smf.phreg(formula, X_train_feats,
                      status=X_train_feats['obito_geral'])
    result = model.fit()

    # Sumário
    print(result.summary())
    print('\n\n')

ESCOLARI
                                       Results: PHReg
Model:                              PH Reg                   Sample size:               35884
Dependent variable:                 meses_diag               Num. events:               17501
Ties:                               Breslow                                                  
---------------------------------------------------------------------------------------------
                                         log HR log HR SE   HR      t    P>|t|  [0.025 0.975]
---------------------------------------------------------------------------------------------
C(ESCOLARI, Treatment(reference=5))[T.1] 0.6647    0.0477 1.9439 13.9428 0.0000 1.7705 2.1343
C(ESCOLARI, Treatment(reference=5))[T.2] 0.4566    0.0353 1.5787 12.9360 0.0000 1.4732 1.6918
C(ESCOLARI, Treatment(reference=5))[T.3] 0.4035    0.0367 1.4971 11.0069 0.0000 1.3933 1.6087
C(ESCOLARI, Treatment(reference=5))[T.4] 0.2390    0.0388 1.2699  6.1585 0.0000 1.1769 1.37

In [ ]:
cont = ['IDADE', 'ANODIAG']

for col in cont:
    print(f'{col}')

    # Fórmula com especificação de categoria de referência
    formula = f"meses_diag ~ {col}"

    # Modelo de Cox (via statsmodels)
    model = smf.phreg(formula, X_train_feats,
                      status=X_train_feats['obito_geral'])
    result = model.fit()

    # Sumário
    print(result.summary())
    print('\n\n')

IDADE
                      Results: PHReg
Model:                 PH Reg        Sample size:    35884
Dependent variable:    meses_diag    Num. events:    17501
Ties:                  Breslow                            
----------------------------------------------------------
      log HR log HR SE   HR      t    P>|t|  [0.025 0.975]
----------------------------------------------------------
IDADE 0.0157    0.0006 1.0159 25.3895 0.0000 1.0146 1.0171
Confidence intervals are for the hazard ratios



ANODIAG
                       Results: PHReg
Model:                  PH Reg         Sample size:     35884
Dependent variable:     meses_diag     Num. events:     17501
Ties:                   Breslow                              
-------------------------------------------------------------
         log HR log HR SE   HR      t    P>|t|  [0.025 0.975]
-------------------------------------------------------------
ANODIAG -0.0056    0.0013 0.9944 -4.4127 0.0000 0.9920 0.9969
Confidence int

In [ ]:
def backward_elimination_lr(data, time_col, event_col, candidate_vars, ref_dict=None, alpha=0.05):
    """
    Backward elimination baseado no teste de razão de verossimilhança.
    Args:
        data (pd.DataFrame): Dados do estudo.
        time_col (str): Nome da coluna de tempo.
        event_col (str): Nome da coluna de status (evento).
        candidate_vars (list): Variáveis candidatas.
        ref_dict (dict): Dicionário com categorias de referência, ex: {'coluna': valor_referência}.
        alpha (float): Nível de significância para o teste de LR.
    Returns:
        list: Variáveis selecionadas.
    """
    selected_vars = candidate_vars[:]

    def create_formula(variables):
        """
        Cria a fórmula considerando as categorias de referência.
        """
        terms = []
        for var in variables:
            if var in ref_dict:  # Adiciona a referência para variáveis categóricas
                terms.append(f"C({var}, Treatment(reference={ref_dict[var]}))")
            else:  # Variáveis contínuas ou sem referência
                terms.append(var)
        return f"{time_col} ~ {' + '.join(terms)}"

    while len(selected_vars) > 0:
        # Modelo completo
        formula_full = create_formula(selected_vars)
        try:
            model_full = smf.phreg(
                formula=formula_full,
                data=data,
                status=data[event_col]
            ).fit()
        except Exception as e:
            print("Erro no modelo completo:")
            print(selected_vars)
            raise e

        # Testar remoção de cada variável
        lr_stats = []
        for var in selected_vars:
            reduced_vars = [v for v in selected_vars if v != var]
            formula_reduced = create_formula(reduced_vars)
            try:
                model_reduced = smf.phreg(
                    formula=formula_reduced,
                    data=data,
                    status=data[event_col]
                ).fit()
            except Exception as e:
                print("Erro no modelo:")
                print(reduced_vars)
                raise e

            # Estatística de razão de verossimilhança
            df_diff = model_full.df_model - model_reduced.df_model
            lr_stat = 2 * (model_full.llf - model_reduced.llf)
            p_value = chi2.sf(lr_stat, df=df_diff)  # df=1 para remover uma variável
            lr_stats.append((var, lr_stat, p_value))

        # Ordenar por p-valor
        lr_stats.sort(key=lambda x: x[2], reverse=True)
        worst_var, worst_lr_stat, worst_p_value = lr_stats[0]

        # Remover variável com maior p-valor se não for significativa
        if worst_p_value > alpha:
            print(f"Removendo variável: {worst_var} (p-valor: {worst_p_value:.4f})")
            selected_vars.remove(worst_var)
        else:
            break

    return selected_vars

In [ ]:
ref_dict = {'ESCOLARI': 5, 'SEXO': 2, 'CATEATEND': 1, 'DIAGPREV': 1, 'TOPO': 11,
            'EC': 0, 'DRS': 1, 'HABILIT2': 2, 'DRS_INST': 1, #'TRATCONS_CAT': 0,
            'DIAGTRAT_CAT': 0}

cols_sig = ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC',
            'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', #'TRATCONS_CAT',
            'DIAGTRAT_CAT']

In [ ]:
final_vars = backward_elimination_lr(data=X_train_feats, time_col="meses_diag",
                                     event_col="obito_geral",
                                     candidate_vars=cols_sig, ref_dict=ref_dict,
                                     alpha=0.05)

print("Variáveis finais selecionadas:", final_vars)

Variáveis finais selecionadas: ['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC', 'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'DIAGTRAT_CAT']


In [ ]:
# Criar a fórmula final com base nas variáveis selecionadas
def create_final_formula(variables, ref_dict):
    """
    Cria a fórmula final considerando categorias de referência.
    Args:
        variables (list): Variáveis selecionadas.
        ref_dict (dict): Dicionário com categorias de referência.
    Returns:
        str: Fórmula para o modelo de Cox.
    """
    terms = []
    for var in variables:
        if var in ref_dict:  # Variáveis categóricas com referência
            terms.append(f"C({var}, Treatment(reference={ref_dict[var]}))")
        else:  # Variáveis contínuas ou sem referência
            terms.append(var)
    return f"meses_diag ~ {' + '.join(terms)}"

In [ ]:
# Gerar a fórmula final
formula_final = create_final_formula(final_vars, ref_dict)

# Ajustar o modelo final
model_final = smf.phreg(formula=formula_final, data=X_train_feats,
                        status=X_train_feats['obito_geral']).fit()

# Exibir sumário do modelo final
print(model_final.summary())

                                           Results: PHReg
Model:                               PH Reg                      Sample size:                  35884
Dependent variable:                  meses_diag                  Num. events:                  17501
Ties:                                Breslow                                                        
----------------------------------------------------------------------------------------------------
                                              log HR log HR SE   HR      t     P>|t|  [0.025  0.975]
----------------------------------------------------------------------------------------------------
C(ESCOLARI, Treatment(reference=5))[T.1]      0.3917    0.0494 1.4795   7.9242 0.0000 1.3429  1.6300
C(ESCOLARI, Treatment(reference=5))[T.2]      0.2811    0.0366 1.3246   7.6801 0.0000 1.2329  1.4231
C(ESCOLARI, Treatment(reference=5))[T.3]      0.2780    0.0378 1.3205   7.3584 0.0000 1.2263  1.4220
C(ESCOLARI, Treatment(reference=5

In [ ]:
X_test_feats[final_vars].columns

Index(['ESCOLARI', 'IDADE', 'SEXO', 'CATEATEND', 'DIAGPREV', 'TOPO', 'EC',
       'ANODIAG', 'DRS', 'HABILIT2', 'DRS_INST', 'DIAGTRAT_CAT'],
      dtype='object')

In [ ]:
pred = pred = model_final.predict(exog=X_test_feats[final_vars])

risk_score = pred.predicted_values

# Conjunto de treino
y_train_surv = Surv.from_arrays(event=X_train_feats["obito_geral"].astype(bool),
                                time=X_train_feats["meses_diag"])

# Conjunto de teste
y_test_surv = Surv.from_arrays(event=X_test_feats["obito_geral"].astype(bool),
                               time=X_test_feats["meses_diag"])

# Se necessário, transformar em vetor 1D
risk_score_ravel = risk_score.ravel()

cindex = concordance_index_censored(X_test_feats["obito_geral"].values.astype(bool),
                                    X_test_feats["meses_diag"],
                                    risk_score)[0]

# C-index IPCW
cindex_ipcw = concordance_index_ipcw(survival_train=y_train_surv,
                                     survival_test=y_test_surv,
                                     estimate=risk_score_ravel)[0]

print(f'C-Index: {cindex:.5f}')
print(f'C-index IPCW: {cindex_ipcw:.5f}')

C-Index: 0.74211
C-index IPCW: 0.73451
